<a href="https://colab.research.google.com/github/joaovictoraraujo231234-maker/Screening-B3/blob/main/Screening_A%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importação de Bibliotecas

In [1]:
!pip install fundamentus
!pip install openpyxl
import fundamentus
import pandas as pd
import time
import yfinance as yf

# Opcional: Configuração para o Pandas mostrar todas as colunas ao imprimir
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Iniciando a extração de dados da B3 (Data do último dia de negociação)...")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is

# Coleta de dados e aplicação de filtros

In [2]:
# =========================================================
# PARÂMETROS DE CONTROLE DO SCREENING
# =========================================================
TETO_PRECO = 20.00
LIQUIDEZ_MINIMA = 5000000
TETO_DIVIDA = 1.5
MIN_ROIC = 0.08
MIN_ROE = 0.08
TETO_PL = 15
DY_MINIMO_LARGE_CAP = 0.05

# =========================================================
# 1. DOWNLOAD DOS DADOS BRUTOS
# =========================================================
df_mercado = fundamentus.get_resultado()

# ==========================================================
# 2. CÁLCULO DO VALOR DE MERCADO E CLASSIFICAÇÃO
# ===========================================================
df_mercado['Valor_Mercado'] = df_mercado['patrliq'] * df_mercado['pvp']

def classificar_tamanho(valor):
    if valor >= 20_000_000_000: return 'Large Cap'
    elif valor >= 5_000_000_000: return 'Mid Cap'
    else: return 'Small Cap'

df_mercado['Classificação'] = df_mercado['Valor_Mercado'].apply(classificar_tamanho)

# ===========================================================
# REGRA ROIC PARA BANCOS
# ===========================================================
def regra_roic_bancos(df, min_roic):
    empresa_operacional = df['roic'] >= min_roic
    excecao_holdings = (df['roic'] >= 0.0) & (df['roic'] < min_roic) & (df['divbpatr'] <= 0.3)
    return empresa_operacional | excecao_holdings
# ==========================================================
# CÁLCULO DY MÉDIO
# ==========================================================
def calcular_dy_medio_5anos(ticker):
    try:
        acao = yf.Ticker(f"{ticker}.SA")
        divs = acao.dividends
        hist = acao.history(period="5y")
        if divs.empty or hist.empty: return 0.0

        divs_anuais = divs.groupby(divs.index.year).sum()
        preco_medio_anual = hist['Close'].groupby(hist.index.year).mean()
        dy_historico = (divs_anuais / preco_medio_anual).dropna()
        return dy_historico.tail(5).mean()
    except:
        return 0.0

# ====================================================================
# 3. IDENTIFICAÇÃO DO "FAST TRACK" (Passe Livre para Vacas Leiteiras)
# ====================================================================
print("Analisando histórico de dividendos (Pode levar alguns segundos)...")

# Lista do BEST para o Bypass (Baseado nos 4 primeiros dígitos do Ticker)
# Isso permite identificar o setor sem precisar mover sua célula de Segmentos
tickers_best = {
    'Bancos': ['BBAS', 'ITUB', 'BBDC', 'SANB', 'BPAC', 'ITSA'],
    'Energia/Saneamento': ['CMIG', 'CPLE', 'SAPR', 'TAEE', 'TRPL', 'ENGI', 'EQTL', 'EGIE', 'CCRO'],
    'Seguros': ['CXSE', 'BBSE', 'PSSA', 'WIZC']
}
# Achata a lista para busca rápida
lista_tickers_com_passe = [t for sublist in tickers_best.values() for t in sublist] + ['KLBN']

def validar_passe_livre(row):
    ticker_base = row.name[:4]
    # Regra: Ser Large Cap do BEST/Klabin + Média de 5 anos >= 5%
    if (ticker_base in lista_tickers_com_passe) and (row['Classificação'] == 'Large Cap'):
        return calcular_dy_medio_5anos(row.name) >= DY_MINIMO_LARGE_CAP
    return False

mask_bypass_dy = df_mercado.apply(validar_passe_livre, axis=1)

# ==========================================================
# 4. APLICAÇÃO DOS FILTROS
# ==========================================================
# A 'Mordaça' que todos devem respeitar (Preço e Liquidez)
mask_global = (df_mercado['cotacao'] <= TETO_PRECO) & (df_mercado['liq2m'] >= LIQUIDEZ_MINIMA)

# A Peneira Rigorosa para Mid/Small e Cíclicas
mask_fundamentos = (
    (df_mercado['divbpatr'] <= TETO_DIVIDA) &
    regra_roic_bancos(df_mercado, MIN_ROIC) &
    (df_mercado['roe'] > MIN_ROE) &
    (df_mercado['mrgliq'] > 0) &
    (df_mercado['pl'] > 0) &
    (df_mercado['pl'] <= TETO_PL)
)

# União: Deve respeitar a Trava Global E (Fundamentos OU Passe Livre)
df_filtrado = df_mercado[mask_global & (mask_fundamentos | mask_bypass_dy)].copy()

# ==========================================================
# 5. REORDENANDO COLUNAS
# ==========================================================
colunas = df_filtrado.columns.tolist()
colunas.remove('Classificação')
colunas.remove('Valor_Mercado')
pos_cotacao = colunas.index('cotacao')
colunas.insert(pos_cotacao, 'Classificação')
colunas.append('Valor_Mercado')
df_filtrado = df_filtrado[colunas]

print(f"Peneira finalizada! {len(df_filtrado)} ações selecionadas.")

Analisando histórico de dividendos (Pode levar alguns segundos)...


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4War

Peneira finalizada! 30 ações selecionadas.


# Classificando o segmento de mercado de acordo com a planilha

In [3]:
# 1. Carrega o arquivo EXCEL diretamente (já que o CSV não foi encontrado)
# O parâmetro usecols="D,F" pega exatamente a Coluna D e Coluna F
# Instalamos openpyxl caso necessário para ler .xlsx
try:
    df_b3 = pd.read_excel('ClassifSetorial.xlsx', skiprows=2, usecols="D,F")
except:
    !pip install openpyxl
    df_b3 = pd.read_excel('ClassifSetorial.xlsx', skiprows=2, usecols="D,F")

df_b3.columns = ['Segmento', 'Codigo_B3']

# 2. Tratamento das células mescladas do Excel
# O método ffill() preenche os vazios "puxando" o nome do segmento da linha de cima
df_b3['Segmento'] = df_b3['Segmento'].ffill()

# 3. Limpeza dos códigos e padronização para texto em maiúsculo
df_b3 = df_b3.dropna(subset=['Codigo_B3'])
df_b3['Codigo_B3'] = df_b3['Codigo_B3'].astype(str).str.strip().str.upper()

# 4. Prepara a base do Fundamentus (df_filtrado) para o cruzamento
df_final = df_filtrado.reset_index().rename(columns={'papel': 'Ticker da Ação'})

# Extraímos os 4 primeiros caracteres do Ticker (ex: de 'ABEV3' para 'ABEV')
# para criar a chave de ligação idêntica à da planilha da B3
df_final['Codigo_B3'] = df_final['Ticker da Ação'].str[:4]

# 5. Realiza o cruzamento (Left Join)
df_final = pd.merge(df_final, df_b3, on='Codigo_B3', how='left')

# Remove a coluna auxiliar do Código de 4 letras que não será mais usada
df_final = df_final.drop(columns=['Codigo_B3'])

print(f"Enriquecimento concluído. {len(df_final)} ações processadas.")

Enriquecimento concluído. 30 ações processadas.


# Formatação do Data Frame

In [4]:
# O dicionário abaixo dita a ordem exata das colunas.
# Repare que 'Segmento' é a segunda chave do dicionário.
colunas_selecionadas = {
    'Ticker da Ação': 'Ticker da Ação',
    'Segmento': 'Segmento',
    'Classificação': 'Tamanho (Market Cap)',
    'cotacao': 'Cotação',
    'pl': 'P/L',
    'pvp': 'P/VP',
    'dy': 'DY',
    'roic': 'ROIC',
    'roe': 'ROE',
    'mrgliq': 'Margem Líquida',
    'divbpatr': 'Dívida Bruta/PL'
}

df_projeto = df_final[list(colunas_selecionadas.keys())].rename(columns=colunas_selecionadas)

# --- APLICAÇÃO DAS FORMATAÇÕES SOLICITADAS ---

# 1. Cotação em formato financeiro (R$)
df_projeto['Cotação'] = df_projeto['Cotação'].apply(lambda x: f"R$ {x:,.2f}".replace('.', ','))

# 2. Indicadores de Rentabilidade em Percentual (%)
colunas_percentuais = ['DY', 'ROIC', 'ROE', 'Margem Líquida']
for col in colunas_percentuais:
    # Caso ocorra algum erro na extração, evitamos que o código quebre
    df_projeto[col] = pd.to_numeric(df_projeto[col], errors='coerce').fillna(0)
    df_projeto[col] = df_projeto[col].apply(lambda x: f"{x:.2%}".replace('.', ','))

# 3. Indicadores de Estrutura de Capital e Preço em formato de Múltiplo (x)
colunas_multiplos = ['P/L', 'Dívida Bruta/PL']
for col in colunas_multiplos:
    df_projeto[col] = pd.to_numeric(df_projeto[col], errors='coerce').fillna(0)
    df_projeto[col] = df_projeto[col].apply(lambda x: f"{x:.1f}x".replace('.', ','))

print("\n" + "="*100)
print("SCREENING FINALIZADO: EMPRESAS SELECIONADAS (COM SEGMENTO B3)")
print("="*100)
print(df_projeto)


SCREENING FINALIZADO: EMPRESAS SELECIONADAS (COM SEGMENTO B3)
   Ticker da Ação                                  Segmento Tamanho (Market Cap)   Cotação    P/L  P/VP      DY    ROIC     ROE Margem Líquida Dívida Bruta/PL
0           BBDC3                                    Bancos            Large Cap  R$ 18,30   7,9x  1.13   7,48%   0,00%  14,25%          0,00%            0,0x
1           CEAB3             Tecidos, Vestuário e Calçados            Small Cap  R$ 13,27   7,0x  1.10   4,22%  14,72%  15,84%          7,35%            0,3x
2           CMIG4                          Energia Elétrica            Large Cap  R$ 13,64   8,0x  1.37   7,58%  10,19%  17,14%         11,46%            0,7x
3           CPLE3                          Energia Elétrica            Large Cap  R$ 16,66  18,5x  2.15   7,34%   8,58%  11,62%         10,29%            0,9x
4           CXSE3                               Seguradoras            Large Cap  R$ 18,92  13,2x  4.19   6,88%  -1,14%  31,67%          0,00%

# EXPORTAÇÃO DO DATAFRAME EM PLANILHA

In [ ]:
# Definindo o nome do arquivo que será salvo
nome_arquivo = 'screening_b3_resultado.xlsx'

# Exportando para Excel
# O parâmetro index=False evita que os números das linhas (0, 1, 2...) virem uma coluna inútil no Excel
df_projeto.to_excel(nome_arquivo, index=False)

print(f"\nSucesso! Os dados foram salvos na planilha: {nome_arquivo}")

# Se preferir exportar em formato CSV (mais leve), você pode usar a linha abaixo em vez da de cima:
# df_projeto.to_csv('screening_b3_resultado.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')


Sucesso! Os dados foram salvos na planilha: screening_b3_resultado.xlsx
